## В этом домашнем задании вы сделаете первые шаги в мире линейной бинарной классификации!

In [2]:
import pandas as pd
import numpy as np
import warnings

#### Задание 1

Мы будем работать с данными **Microsoft Malware Detection**

Таргетом будет последний столбец `HasDetection`, который принимает значения $\{0,\, 1\}$ в случае отсутствия или наличия вируса на компьютере соответственно. Признаками будут выступать всевозможные характеристики компьютера.

In [3]:
data = pd.read_csv('train.csv')

/tmp/ipykernel_24309/2815211952.py:1: DtypeWarning: Columns (0: PuaMode) have mixed types. Specify dtype option on import or set low_memory=False.
  data = pd.read_csv('train.csv')


In [4]:
data.head()

,MachineIdentifier,ProductName,EngineVersion,AppVersion,AvSigVersion,IsBeta,RtpStateBitfield,IsSxsPassiveMode,DefaultBrowsersIdentifier,AVProductStatesIdentifier,...,Census_FirmwareVersionIdentifier,Census_IsSecureBootEnabled,Census_IsWIMBootEnabled,Census_IsVirtualDevice,Census_IsTouchEnabled,Census_IsPenCapable,Census_IsAlwaysOnAlwaysConnectedCapable,Wdft_IsGamer,Wdft_RegionIdentifier,HasDetections
0,0000028988387b115f69f31a3bf04f09,win8defender,1.1.15100.1,4.18.1807.18075,1.273.1735.0,0,7.0,0,NaN,53447.0,...,36144.0,0,NaN,0.0,0,0,0.0,0.0,10.0,0
1,000007535c3f730efa9ea0b7ef1bd645,win8defender,1.1.14600.4,4.13.17134.1,1.263.48.0,0,7.0,0,NaN,53447.0,...,57858.0,0,NaN,0.0,0,0,0.0,0.0,8.0,0
2,000007905a28d863f6d0d597892cd692,win8defender,1.1.15100.1,4.18.1807.18075,1.273.1341.0,0,7.0,0,NaN,53447.0,...,52682.0,0,NaN,0.0,0,0,0.0,0.0,3.0,0
3,00000b11598a75ea8ba1beea8459149f,win8defender,1.1.15100.1,4.18.1807.18075,1.273.1527.0,0,7.0,0,NaN,53447.0,...,20050.0,0,NaN,0.0,0,0,0.0,0.0,3.0,1
4,000014a5f00daa18e76b81417eeb99fc,win8defender,1.1.15100.1,4.18.1807.18075,1.273.1379.0,0,7.0,0,NaN,53447.0,...,19844.0,0,0.0,0.0,0,0,0.0,0.0,1.0,1


Удалите константные признаки и признаки `ProductName` `MachineIdentifier`

In [5]:
data = data.drop(columns=data.columns[data.nunique() == 1], errors='ignore')
data = data.drop(columns=['ProductName', 'MachineIdentifier'], errors='ignore')

Посмотрите на соотношение классов в таргете. Все ли хорошо?

In [6]:
print(sum(data['HasDetections'] == 1), '- positive class,')
print(sum(data['HasDetections'] == 0), '- negative class')

100060 - positive class,
99940 - negative class


Ответьте на вопрос: почему с вашей точки зрения важно иметь представление о балансе классов в ваших данных?

Избавьтесь от пропусков в данных! 

Новый для нас прием: признаки с более чем половиной пропусков следует удалить.

Согласитесь, если в вашей колонке среди 100 объектов всего лишь у 2 есть какое-то непропущенное значение, странно все остальные заполнять средним от этих двух чисел. Такие "редкие" признаки лучше вообще опустить!


В категориальных колонках заменим отсутствующую категорию просто некоторой новой и назовем ее `NaN`.

А в числовых, ради разнообразия, заполним пропуски медианным значением.

In [7]:
columns_to_drop = data.columns[data.isna().sum(axis=0) > (len(data.index)) / 2]
print(columns_to_drop)
data = data.drop(columns=columns_to_drop, errors='ignore')

num_columns = data.select_dtypes(include='number').columns
data[num_columns] = data[num_columns].fillna(data[num_columns].median())

str_columns = data.select_dtypes(include=['object', 'str']).columns
data[str_columns] = data[str_columns].fillna('NaN')

# Проверка на наличие пропусков
if data.isna().any(axis=1).any():
    print('В данных еще остались пропуски')
else:
    print('В данных не осталось пропусков')

Index(['DefaultBrowsersIdentifier', 'Census_ProcessorClass',
       'Census_InternalBatteryType', 'Census_IsFlightingInternal',
       'Census_ThresholdOptIn'],
      dtype='str')
В данных не осталось пропусков


Создайте копию полученного датафрейма и положите ее в переменную data_2. Понадобится в следующих заданиях.

In [8]:
data_2 = data.copy()

Так же поработаем над всеми категориальными колонкам перед запуском непосредственно моделей.

Провернем самый базовый и наглый метод - несмотря на количество уникальных значений в каждой категории, просто применим ко всей категориальной части датасета `OneHotEncoding`

In [9]:
columns_to_encode = data.select_dtypes(include=['object', 'str']).columns

data = pd.get_dummies(data, columns=columns_to_encode)

###  Разделим выборку на тренировочную и тестовую

P.S. в задачах классификации, как и в задаче регрессии, можно использовать технологию Кросс-Валидации. Например, по одному из двух следующих сценариев:

1) Отделить валидацию и тест, произвести подбор лучшей модели с помощью `K-Fold` на валидации, финально обучить выбранную модель на всей валидации и замерить качество на заранее отложенном финальном тесте!

2) Всю выборку назвать валидационной и на ней применить `K-Fold` без финального замера.

В этой домашней работе попросим Вас быть еще проще! :)
Реализуем просто технологию отложенной выборки в пропорции 3:1

In [10]:
from sklearn.model_selection import train_test_split 

X = data.drop(columns=['HasDetections'])
y = data['HasDetections']

X_train, X_test, y_train, y_test = train_test_split(X, y,
                                                    test_size=0.25,
                                                    shuffle=True,
                                                    random_state=1)

Соберите `Pipeline`, реализовав в нем 2 шага: стандартизация данных через `StandardScaler` и обучение логистической регрессии с помощью `LogisticRegression`, положите результаты в переменную `pipe`, а в классе модели `LogisticRegression` укажите параметр `penalty='none'`

In [11]:
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline

pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('model', LogisticRegression(C=np.inf)) # вместо penalty=None
])

Чтобы замерить качество работы такой модели на трейне и на тесте воспользуемся функцией `cross_validate`

Вопрос: что передавать ей в параметр cv? Ведь мы уже разделили нашу выборку на трейн и тест (имеем всего 1 fold). Для этого можно просто передать список от кортежа, содержащего индексы тренировочных и тестовых объектов.

In [11]:
from sklearn.model_selection import cross_validate
import datetime

custom_cv = [(X_train.index.to_list(), X_test.index.to_list())]

begin_time = datetime.datetime.now()

cv_result_pipe = cross_validate(pipe, X, y, scoring='accuracy',
                                cv=custom_cv, return_train_score=True, n_jobs=-1)


print(f"Accuracy на трейне: {np.mean(cv_result_pipe['train_score']).round(3)}")
print(f"Accuracy на тесте: {np.mean(cv_result_pipe['test_score']).round(3)}")

print(f"Время работы алгоритма: {datetime.datetime.now() - begin_time}")

Accuracy на трейне: 0.651
Accuracy на тесте: 0.623
Время работы алгоритма: 0:04:21.973957


Что можете сказать про время работы алгоритма?

Очевидно, оно достаточно большие. Уж тем более для линейных моделей.

Такое как раз-таки происходит из-за того, что количество фичей, который мы передали нашей модели - гигантское! Классу требуется много времени и памяти, чтобы обработать датасет.

Поэтому те колонки, в которых количество уникальных категорий превышает какое-то адекватное число, следует кодировать иначе, нежели с помощью технологии `One-Hot-Encoding`.

Теперь вы верите, что более умные кодировки зачастую прям необходимы! Раньше мы этот факт не демонстрировали!

Дело еще вот в чем: в классе `LogisticRegression`, как и, например, `Lasso`, есть параметр, ограничивающий максимальное количество итераций во время обучения модели. Так, если данных много и итераций тоже ожидается большое число, найденная разделяющая гиперплоскость может оказаться не самой лучшей, так как наш алгоритм (будь то градиентный спуск или любой иной) просто 'не доползет'. 

#### Задание 2

Теперь попробуем другой метод кодирования категориальных колонок, а именно счётчики.
Построем ту же модель и на том же разделении, просто заново иначе переобработаем датасет. 

Для тех категориальных признаков, у которых количество уникальных значений в колоночках больше 5, применим `MeanTargetEncoding`.

Для остальных оставим любимый `OneHotEncoding` (как делали на практике и в предыдущем уроке).

In [12]:
str_columns = data_2.select_dtypes(include=['object', 'str']).columns
numeric_columns = data.select_dtypes(include=['int64', 'float64']).columns.tolist()

columns_to_ohe = str_columns[data_2[str_columns].nunique() <= 5]
columns_to_mte = str_columns[data_2[str_columns].nunique() > 5]

for column_to_mte in columns_to_mte:
    mean_target = data_2.groupby(column_to_mte)['HasDetections'].mean()
    data_2[column_to_mte] = data_2[column_to_mte].map(mean_target)

data_2 = pd.get_dummies(data_2, columns=columns_to_ohe)

In [13]:
X_2 = data_2.drop(columns=['HasDetections'])
y_2 = data_2['HasDetections']

Опять обучим модель, пока что без изменений! Скажите, стало ли быстрее? А что с качеством?

In [14]:
from sklearn.model_selection import cross_validate, train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
import datetime

X_2_train, X_2_test, y_2_train, y_2_test = train_test_split(X_2, y_2,
                                                    test_size=0.25,
                                                    shuffle=True,
                                                    random_state=1)

custom_cv = [(X_2_train.index.to_list(), X_2_test.index.to_list())]

pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('model', LogisticRegression(C=np.inf))
])

begin_time = datetime.datetime.now()

cv_result_pipe = cross_validate(pipe, X_2, y_2, scoring='accuracy',
                                cv=custom_cv, return_train_score=True)


print(f"Accuracy на трейне: {np.mean(cv_result_pipe['train_score']).round(3)}")
print(f"Accuracy на тесте: {np.mean(cv_result_pipe['test_score']).round(3)}")

print(f"Время работы алгоритма: {datetime.datetime.now() - begin_time}")

# Accuracy на тесте возросло, а на трейне стало чуть меньше. Время работы намного быстрее

Accuracy на трейне: 0.638
Accuracy на тесте: 0.638
Время работы алгоритма: 0:00:03.064106


#### Задание 3: Регуляризация

Как и в моделях регрессии, решая задачу классификации, можем штрафовать минимизируемый функционал за большие веса, добавив к нему L1 или L2 норму весов (все как раньше!).

Для этого в изначальном классе `LogisticRegression` изменить параметр `penalty` на l1 или l2 соответственно. Выберите второй вариант! Можно воспользоваться методом `set_params` и применить его к `pipe`.

In [15]:
pipe.set_params(model__l1_ratio=0) 
print(pipe.get_params()['model__l1_ratio']) 

0


Теперь наша модель будет строить логистическую регрессию с L2 регуляризатором! Помним, что у регуляризируемых моделей есть гиперпараметр, контролирующий силу регуляризации, который выбирается ДО запуска метода fit, то есть заранее, когда модель еще не обучена. 

Конечно же, выбор этого параметра влияет итоговые результаты. Хочется поперебирать различные параметры регуляризации и найти такой, при котором качество на тесте окажется лучшим! 

Сгенерируем массив гиперпараметра, которые планируем перебрать:

In [16]:
alphas = np.linspace(0.01, 100, 100)
alphas

array([1.000e-02, 1.020e+00, 2.030e+00, 3.040e+00, 4.050e+00, 5.060e+00,
       6.070e+00, 7.080e+00, 8.090e+00, 9.100e+00, 1.011e+01, 1.112e+01,
       1.213e+01, 1.314e+01, 1.415e+01, 1.516e+01, 1.617e+01, 1.718e+01,
       1.819e+01, 1.920e+01, 2.021e+01, 2.122e+01, 2.223e+01, 2.324e+01,
       2.425e+01, 2.526e+01, 2.627e+01, 2.728e+01, 2.829e+01, 2.930e+01,
       3.031e+01, 3.132e+01, 3.233e+01, 3.334e+01, 3.435e+01, 3.536e+01,
       3.637e+01, 3.738e+01, 3.839e+01, 3.940e+01, 4.041e+01, 4.142e+01,
       4.243e+01, 4.344e+01, 4.445e+01, 4.546e+01, 4.647e+01, 4.748e+01,
       4.849e+01, 4.950e+01, 5.051e+01, 5.152e+01, 5.253e+01, 5.354e+01,
       5.455e+01, 5.556e+01, 5.657e+01, 5.758e+01, 5.859e+01, 5.960e+01,
       6.061e+01, 6.162e+01, 6.263e+01, 6.364e+01, 6.465e+01, 6.566e+01,
       6.667e+01, 6.768e+01, 6.869e+01, 6.970e+01, 7.071e+01, 7.172e+01,
       7.273e+01, 7.374e+01, 7.475e+01, 7.576e+01, 7.677e+01, 7.778e+01,
       7.879e+01, 7.980e+01, 8.081e+01, 8.182e+01, 

Чтобы отобрать среди данного массива гиперпараметров лучший, воспользуйтесь конструкцией `GridSearchCV` из `sklearn`

In [17]:
from sklearn.model_selection import GridSearchCV

pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('model', LogisticRegression(l1_ratio=0, max_iter=1000))   # L2 + достаточно итераций
])

grid = GridSearchCV(pipe, param_grid={'model__C': alphas},
                    scoring='accuracy', cv=custom_cv)          # разбиение из задания
grid.fit(X_2, y_2)

print(grid.best_params_)
print(round(grid.best_score_, 3))

{'model__C': np.float64(0.01)}
0.638


#### Задание 4: Бонус

Как вы знаете, подбор признаков является одной из самых важных частей машинного обучения. Сейчас вы попробуете на основе имеющихся признаков сгенерировать новые. В качестве новых признаков будем использовать мономы 2 степени. Можно использовать регуляризацию различного рода, выбор энкодера тоже за вами. Ваша задача - добиться качества `0.65` на тестовой выборке

In [19]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import PolynomialFeatures
from sklearn.model_selection import GridSearchCV

# НЕ режем data_2 — берём все признаки (numeric + MTE + OHE)
X_2 = data_2.drop(columns=['HasDetections'])
y_2 = data_2['HasDetections']

# Специально обработаем признак Census_InternalBatteryNumberOfCharges
# Этот признак отвечает за количество зарядов батареи. 
# Случай, когда батареи нет (десктоп), обозначен как 4.294967e+09
# Не обработать этот случай нельзя, так как он будет мешать нормализации признака. 
# Это число сильно искажает распределение признака
BATTERY_SENTINEL = int(X_2['Census_InternalBatteryNumberOfCharges'].max())

X_2['IsDesktop'] = X_2['Census_InternalBatteryNumberOfCharges'] >= BATTERY_SENTINEL
real_charges = X_2.loc[~X_2['IsDesktop'], 'Census_InternalBatteryNumberOfCharges'].median()
X_2['BatteryChargeCount'] = np.where(X_2['IsDesktop'] == 1, real_charges, X_2['Census_InternalBatteryNumberOfCharges'])
X_2 = X_2.drop(columns=['Census_InternalBatteryNumberOfCharges'])

columns_to_poly = [
    'Census_ProcessorCoreCount', 
    'Census_PrimaryDiskTotalCapacity', 
    'Census_SystemVolumeTotalCapacity', 
    'Census_TotalPhysicalRAM', 
    'Census_InternalPrimaryDiagonalDisplaySizeInInches', 
    'Census_InternalPrimaryDisplayResolutionHorizontal',
    'Census_InternalPrimaryDisplayResolutionVertical',
    'BatteryChargeCount'
]

columns_to_not_poly  = [c for c in numeric_columns if c != 'HasDetections' and c in X_2.columns and c not in columns_to_poly]

# Необходимо обработать также числовые признаки-идентификаторы
# Это категориальные признаки, которые закодированы числами
# Проблема с ними заключается в том, что они учитывают порядок, а здесь нам этот порядок не нужен
# Они учитывают порядок, но и мало того, они еще неправильно учитывают пропорции вклада в общую взвешенную сумму
# Пример:
# Вклад города питер с числовой категорией 5 в 100 раз больше чем вклад города Саратов с категорией 4, 
# Но Москва с категорией 6 дает вклад в 10 раз больше чем Питер и в 1000 раз больше чем Саратов.
# Нельзя подобрать такой вес, который закодирует такую закономерность
# У нас все равно получатся вот такие вклады в взвешенную сумму:
# 4w, 5w, 6w
# И это еще ситуация, в которых вклад строго не убывает при возрастающей категории
# А могло быть например так:
# Категория 6 могла бы давать вклад больше чем категория 5, но категория 4 могла бы давать вклад больше чем категория 5
# И подобрать вес который учитывал бы такую закономерность невозможно
# Поэтому применим Mean Target Encoding (MTE) к этим признакам
# Мы здесь считаем MTE по всей выборке, а не только по трейну, так как в задании 2 мы считали MTE по всей выборке до сплита.

not_binary_columns = [numeric_column for numeric_column in columns_to_not_poly if not (set(X_2[numeric_column].unique()) <= {0, 1})]
for column_to_mte in not_binary_columns:
    mean_target = y_2.groupby(X_2[column_to_mte]).mean()
    X_2[column_to_mte] = X_2[column_to_mte].map(mean_target)

rest_feats = [c for c in X_2.columns if c not in columns_to_not_poly and c not in columns_to_poly]  

X_2_train, X_2_test, y_2_train, y_2_test = train_test_split(
    X_2, y_2, test_size=0.25, shuffle=True, random_state=1)

custom_cv = [(X_2_train.index.to_list(), X_2_test.index.to_list())]

features = ColumnTransformer([
    ('poly', PolynomialFeatures(degree=2, include_bias=False), columns_to_poly),
    ('not_poly', 'passthrough', columns_to_not_poly),
    ('rest', 'passthrough', rest_feats),
])

pipe = Pipeline([
    ('features', features),
    ('scaler', StandardScaler()),
    ('model', LogisticRegression(l1_ratio=1, solver='saga', max_iter=2000, tol=1e-3, random_state=1)),
], memory='./sklearn_cache')

grid = GridSearchCV(pipe, {'model__C': np.logspace(-3, 2, 4)},
                    cv=custom_cv, scoring='accuracy', verbose=2)
grid.fit(X_2, y_2)

print("Best C:", grid.best_params_['model__C'])
print("Best accuracy on test:", round(grid.best_score_, 3))

Fitting 1 folds for each of 4 candidates, totalling 4 fits
[CV] END .....................................model__C=0.001; total time=  25.0s
[CV] END ......................model__C=0.046415888336127795; total time=  27.1s
[CV] END ........................model__C=2.1544346900318843; total time=  27.9s
[CV] END .....................................model__C=100.0; total time=  27.5s
Best C: 2.1544346900318843
Best accuracy on test: 0.746
